# Interactive B-spline Explorer — chordal vs model, 2D/3D, choose knots & points

Everything from the training notebook, plus a live control panel:

* **Dimension** — switch between 2D and 3D
* **Interior knots** — pick 1…10 (the point count is knots + 2)
* **Data points** — edit the coordinates (one point per line) or hit *Randomize*
* **Side-by-side** — left = chordal heuristic, right = your model (or the energy-optimized
  target), with the **curvature κ shown on top** and the ratio in the header

Run cells 1–3 once, then use the panel in cell 4. Optionally upload a trained checkpoint
to show *your model* on the right instead of the energy-optimized target.


## 1 · Setup (writes the modules to disk)


In [ ]:
import torch
torch.set_default_dtype(torch.float64)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
bspline3d_src = r'''
"""
bspline3d.py  —  dimension-general (2D/3D/nD) differentiable B-spline
interpolation + curvature bending energy, in pure PyTorch.

The ONLY mathematics that differs from the 2D code is the curvature:
a space curve uses the full vector cross product  kappa = ||C' x C''|| / ||C'||^3 ,
which reduces exactly to the 2D scalar formula when z = 0.

Everything is differentiable w.r.t. the interior knots, so the bending energy
can be used directly as a self-supervised Phase-2 loss.
"""
import torch

P_DEG = 3
_EPS = 1e-12


# ----------------------------------------------------------------------
# Knot / parameter construction
# ----------------------------------------------------------------------
def clamped_knots(u_int):
    """u_int: (..., K) interior knots in (0,1) -> full clamped cubic vector (..., K+8)."""
    *lead, K = u_int.shape
    z = u_int.new_zeros(*lead, P_DEG + 1)
    o = u_int.new_ones(*lead, P_DEG + 1)
    return torch.cat([z, u_int, o], dim=-1)


def data_params(u_int):
    """Data points are interpolated at t = [0, u_1, ..., u_K, 1]."""
    *lead, K = u_int.shape
    z = u_int.new_zeros(*lead, 1)
    o = u_int.new_ones(*lead, 1)
    return torch.cat([z, u_int, o], dim=-1)


# ----------------------------------------------------------------------
# Differentiable Cox-de Boor basis (single knot vector, batch of params)
# ----------------------------------------------------------------------
def _safe_div(num, den):
    return num / torch.where(den.abs() < _EPS, torch.ones_like(den), den) \
        * (den.abs() >= _EPS)


def basis_matrix(U, t, p=P_DEG, deriv=0):
    """
    U : (M,) knot vector (1D)
    t : (T,) parameters
    returns (T, n_ctrl) matrix of N_{i,p}^{(deriv)}(t_j), differentiable w.r.t. U and t.
    Ordinary recursion for degrees 1..(p-deriv); derivative recursion for the top `deriv` degrees.
    """
    M = U.shape[0]
    n_ctrl = M - p - 1
    Ul = U[:-1].unsqueeze(0)
    Ur = U[1:].unsqueeze(0)
    tt = t.unsqueeze(1)
    end = U[-1]
    ge = tt >= Ul
    lt = tt < Ur
    at_end = (tt >= end) & (Ur >= end)
    N = (ge & (lt | at_end)).to(U.dtype)            # degree 0

    for q in range(1, p + 1):
        ncol = M - q - 1
        Ni   = N[:, :ncol]
        Ni1  = N[:, 1:ncol + 1]
        Ui   = U[:ncol].unsqueeze(0)
        Uiq  = U[q:q + ncol].unsqueeze(0)
        Ui1  = U[1:1 + ncol].unsqueeze(0)
        Uiq1 = U[q + 1:q + 1 + ncol].unsqueeze(0)
        d1 = Uiq - Ui
        d2 = Uiq1 - Ui1
        if q <= p - deriv:
            term1 = _safe_div(tt - Ui, d1) * Ni
            term2 = _safe_div(Uiq1 - tt, d2) * Ni1
        else:
            qf = float(q)
            term1 = _safe_div(qf * Ni, d1)
            term2 = _safe_div(qf * Ni1, d2)
            N = term1 - term2
            continue
        N = term1 + term2
    return N


# ----------------------------------------------------------------------
# Collocation matrix and control-point solve (any dimension d)
# ----------------------------------------------------------------------
def build_collocation(u_int):
    """u_int: (K,) -> A (N+2, N+2), full knot vector U."""
    U = clamped_knots(u_int)
    tbar = data_params(u_int)                  # (N,)
    N = tbar.shape[0]
    rows_interp = basis_matrix(U, tbar, P_DEG, deriv=0)    # (N, N+2)
    ends = torch.stack([u_int.new_zeros(1).squeeze(0),
                        u_int.new_ones(1).squeeze(0)])      # t=0, t=1
    rows_end = basis_matrix(U, ends, P_DEG, deriv=2)        # (2, N+2)  C''=0 rows
    A = torch.cat([rows_interp, rows_end], dim=0)           # (N+2, N+2)
    return A, U


def solve_control_points(D, u_int):
    """
    D     : (N, d) data points (d = 2 or 3 or anything)
    u_int : (K,) interior knots, K = N-2
    returns control points P : (N+2, d)
    """
    A, U = build_collocation(u_int)
    N, d = D.shape
    rhs = torch.cat([D, D.new_zeros(2, d)], dim=0)          # (N+2, d)
    P = torch.linalg.solve(A, rhs)                          # differentiable solve
    return P, U


# ----------------------------------------------------------------------
# Curve derivatives and curvature bending energy
# ----------------------------------------------------------------------
def eval_derivs(U, P, t):
    """Return C, C', C'' at parameters t. P:(n_ctrl,d). Each output (T,d)."""
    B0 = basis_matrix(U, t, P_DEG, deriv=0)
    B1 = basis_matrix(U, t, P_DEG, deriv=1)
    B2 = basis_matrix(U, t, P_DEG, deriv=2)
    return B0 @ P, B1 @ P, B2 @ P


def _cross_norm(Cp, Cpp):
    """||C' x C''|| that works for d=2 (scalar cross) and d=3 (vector cross)."""
    d = Cp.shape[-1]
    if d == 2:
        return (Cp[..., 0] * Cpp[..., 1] - Cp[..., 1] * Cpp[..., 0]).abs()
    elif d == 3:
        return torch.linalg.norm(torch.cross(Cp, Cpp, dim=-1), dim=-1)
    else:
        # general nD: ||C'||^2||C''||^2 - (C'.C'')^2  (Gram determinant)
        a = (Cp * Cp).sum(-1)
        b = (Cpp * Cpp).sum(-1)
        c = (Cp * Cpp).sum(-1)
        return torch.sqrt(torch.clamp(a * b - c * c, min=0.0))


def bending_energy(D, u_int, n_quad=100):
    """
    Total squared-curvature bending energy   E = int_0^1 kappa^2 ||C'|| dt
    D:(N,d), u_int:(K,). Differentiable w.r.t. u_int. Works for d=2 and d=3.
    """
    P, U = solve_control_points(D, u_int)
    # midpoint quadrature on (0,1)
    t = (torch.arange(n_quad, dtype=D.dtype, device=D.device) + 0.5) / n_quad
    C0, Cp, Cpp = eval_derivs(U, P, t)
    speed = torch.linalg.norm(Cp, dim=-1)                  # ||C'||
    kappa = _cross_norm(Cp, Cpp) / (speed ** 3 + _EPS)     # curvature
    integrand = kappa ** 2 * speed                         # kappa^2 ||C'||
    return integrand.mean()                                 # (b-a)=1 -> mean == integral


# ----------------------------------------------------------------------
# Chordal heuristic (any dimension)
# ----------------------------------------------------------------------
def chordal_knots(D):
    """D:(N,d) -> interior knots (K,) by cumulative chord length."""
    diffs = torch.linalg.norm(D[1:] - D[:-1], dim=-1)
    cum = torch.cumsum(diffs, 0)
    total = cum[-1]
    frac = cum / (total + _EPS)
    return frac[:-1].clone()


def safe_curvature(D, u_int, n_quad=100):
    """Non-differentiable scalar energy with degeneracy guards (for eval)."""
    u = torch.as_tensor(u_int, dtype=D.dtype, device=D.device)
    su, _ = torch.sort(u)
    spacing_ok = (su.numel() < 2) or ((su[1:] - su[:-1]).min() >= 1e-4)
    if (not spacing_ok) or su.min() <= 1e-4 or su.max() >= 1 - 1e-4:
        return float('inf')
    try:
        with torch.no_grad():
            e = bending_energy(D, su, n_quad)
        v = float(e)
        return v if v == v and v != float('inf') else float('inf')
    except Exception:
        return float('inf')


# ----------------------------------------------------------------------
# Batched versions (all samples in a group share the same N) — for GPU training
# ----------------------------------------------------------------------
def basis_matrix_batch(U, t, p=P_DEG, deriv=0):
    """U:(G,M) batch of knot vectors, t:(T,) shared params -> (G,T,n_ctrl)."""
    G, M = U.shape
    Ul = U[:, :-1].unsqueeze(1)          # (G,1,M-1)
    Ur = U[:, 1:].unsqueeze(1)
    tt = t.view(1, -1, 1)                 # (1,T,1)
    end = U[:, -1].view(G, 1, 1)
    ge = tt >= Ul
    lt = tt < Ur
    at_end = (tt >= end) & (Ur >= end)
    N = (ge & (lt | at_end)).to(U.dtype)  # (G,T,M-1)
    for q in range(1, p + 1):
        ncol = M - q - 1
        Ni  = N[:, :, :ncol]
        Ni1 = N[:, :, 1:ncol + 1]
        Ui   = U[:, :ncol].unsqueeze(1)
        Uiq  = U[:, q:q + ncol].unsqueeze(1)
        Ui1  = U[:, 1:1 + ncol].unsqueeze(1)
        Uiq1 = U[:, q + 1:q + 1 + ncol].unsqueeze(1)
        d1 = Uiq - Ui
        d2 = Uiq1 - Ui1
        if q <= p - deriv:
            t1 = _safe_div(tt - Ui, d1) * Ni
            t2 = _safe_div(Uiq1 - tt, d2) * Ni1
            N = t1 + t2
        else:
            qf = float(q)
            N = _safe_div(qf * Ni, d1) - _safe_div(qf * Ni1, d2)
    return N



def basis_diag(U, tvals, p=P_DEG, deriv=0):
    """U:(G,M), tvals:(G,) per-sample scalar params -> (G,n_ctrl)."""
    G, M = U.shape
    Ul = U[:, :-1]; Ur = U[:, 1:]
    tt = tvals.unsqueeze(1)
    end = U[:, -1:].clone()
    ge = tt >= Ul; lt = tt < Ur
    at_end = (tt >= end) & (Ur >= end)
    N = (ge & (lt | at_end)).to(U.dtype)
    for q in range(1, p + 1):
        ncol = M - q - 1
        Ni  = N[:, :ncol]; Ni1 = N[:, 1:ncol + 1]
        Ui   = U[:, :ncol]; Uiq  = U[:, q:q + ncol]
        Ui1  = U[:, 1:1 + ncol]; Uiq1 = U[:, q + 1:q + 1 + ncol]
        d1 = Uiq - Ui; d2 = Uiq1 - Ui1
        if q <= p - deriv:
            N = _safe_div(tt - Ui, d1) * Ni + _safe_div(Uiq1 - tt, d2) * Ni1
        else:
            qf = float(q); N = _safe_div(qf * Ni, d1) - _safe_div(qf * Ni1, d2)
    return N


def bending_energy_batch(D, u_int, n_quad=100):
    """
    D:(G,N,d) same N across the group, u_int:(G,K) -> (G,) energies. Differentiable.
    """
    G, Nn, d = D.shape
    K = u_int.shape[1]
    z = u_int.new_zeros(G, P_DEG + 1); o = u_int.new_ones(G, P_DEG + 1)
    U = torch.cat([z, u_int, o], dim=1)                       # (G, K+8)
    tbar = torch.cat([u_int.new_zeros(G, 1), u_int, u_int.new_ones(G, 1)], dim=1)  # (G,N)
    # collocation rows must be evaluated at per-sample params -> loop only over the
    # N interpolation rows is avoided by building A per sample via batched basis at tbar.
    # tbar differs per sample, so evaluate basis at each sample's own params:
    A_rows = []
    # interpolation rows: for each data-parameter position j, basis at tbar[:,j]
    # do it vectorised over j using a (G,N,n_ctrl) gather
    n_ctrl = Nn + 2
    Aint = torch.stack([basis_diag(U, tbar[:, j], P_DEG, 0) for j in range(Nn)], dim=1)  # (G,N,n_ctrl)
    z_end = basis_diag(U, u_int.new_zeros(G), P_DEG, deriv=2)  # (G,n_ctrl)
    o_end = basis_diag(U, u_int.new_ones(G),  P_DEG, deriv=2)
    Aend = torch.stack([z_end, o_end], dim=1)                 # (G,2,n_ctrl)
    A = torch.cat([Aint, Aend], dim=1)                        # (G,N+2,N+2)
    rhs = torch.cat([D, D.new_zeros(G, 2, d)], dim=1)          # (G,N+2,d)
    P = torch.linalg.solve(A, rhs)                            # (G,N+2,d)
    tq = (torch.arange(n_quad, dtype=D.dtype, device=D.device) + 0.5) / n_quad
    B0 = basis_matrix_batch(U, tq, P_DEG, 0)                  # (G,T,n_ctrl)
    B1 = basis_matrix_batch(U, tq, P_DEG, 1)
    B2 = basis_matrix_batch(U, tq, P_DEG, 2)
    Cp  = torch.bmm(B1, P)                                     # (G,T,d)
    Cpp = torch.bmm(B2, P)
    speed = torch.linalg.norm(Cp, dim=-1)
    if d == 2:
        cn = (Cp[..., 0]*Cpp[..., 1] - Cp[..., 1]*Cpp[..., 0]).abs()
    else:
        cn = torch.linalg.norm(torch.cross(Cp, Cpp, dim=-1), dim=-1)
    kappa = cn / (speed**3 + _EPS)
    return (kappa**2 * speed).mean(dim=1)                     # (G,)

'''
open('bspline3d.py','w').write(bspline3d_src)
kt3d_src = r'''
"""
knot_transformer_3d.py — KnotTransformer adapted for d-dimensional points
(works for d=2 or d=3). The ONLY change from the 2D model is the input:
the geometric-feature extractor consumes d coordinates instead of 2.
Encoder, decoder, knot-queries and knot-head are dimension-independent
because knots live in 1D parameter space.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F


def geometric_features(points, mask=None):
    """
    points : (B, N, d)   padded point sequences
    mask   : (B, N) bool, True = real point (optional)
    returns (B, N, d+3): [coords(d), dist_prev, dist_next, turning_angle]
    All terms are Euclidean / dot-product based, so they work for any d.
    """
    B, N, d = points.shape
    prev = torch.zeros_like(points); prev[:, 1:] = points[:, :-1]
    nxt  = torch.zeros_like(points); nxt[:, :-1] = points[:, 1:]
    dist_prev = torch.linalg.norm(points - prev, dim=-1, keepdim=True)
    dist_next = torch.linalg.norm(nxt - points, dim=-1, keepdim=True)
    v_in  = points - prev
    v_out = nxt - points
    cos = (v_in * v_out).sum(-1) / (
        torch.linalg.norm(v_in, dim=-1) * torch.linalg.norm(v_out, dim=-1) + 1e-9)
    angle = torch.acos(cos.clamp(-1 + 1e-6, 1 - 1e-6)).unsqueeze(-1)
    feats = torch.cat([points, dist_prev, dist_next, angle], dim=-1)
    return torch.nan_to_num(feats, 0.0)


class KnotTransformer3D(nn.Module):
    def __init__(self, dim=3, d_model=64, nhead=4,
                 num_encoder_layers=3, num_decoder_layers=3,
                 dim_feedforward=256, dropout=0.1,
                 max_points=12, max_knots=8):
        super().__init__()
        self.dim = dim
        self.max_points = max_points
        self.max_knots = max_knots
        in_feat = dim + 3                                  # coords + 3 geom features
        self.input_proj = nn.Linear(in_feat, d_model)
        self.pos_embed  = nn.Parameter(torch.randn(max_points, d_model) * 0.02)
        enc = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward,
                                         dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc, num_encoder_layers)
        dec = nn.TransformerDecoderLayer(d_model, nhead, dim_feedforward,
                                         dropout, batch_first=True)
        self.decoder = nn.TransformerDecoder(dec, num_decoder_layers)
        self.knot_queries = nn.Parameter(torch.randn(max_knots, d_model) * 0.02)
        self.knot_head = nn.Sequential(
            nn.Linear(d_model, d_model // 2), nn.GELU(),
            nn.Linear(d_model // 2, 1))

    def forward(self, points, point_mask, num_knots):
        """
        points     : (B, N, dim) padded
        point_mask : (B, N) bool, True = PAD (Transformer convention)
        num_knots  : (B,) int, how many interior knots each sample needs
        returns (B, max_knots) knot values in (0,1); use first num_knots[b] per sample.
        """
        B, N, _ = points.shape
        real = ~point_mask
        feats = geometric_features(points, real)
        x = self.input_proj(feats) + self.pos_embed[:N].unsqueeze(0)
        memory = self.encoder(x, src_key_padding_mask=point_mask)
        q = self.knot_queries.unsqueeze(0).expand(B, -1, -1)
        dec = self.decoder(q, memory, memory_key_padding_mask=point_mask)
        raw = self.knot_head(dec).squeeze(-1)              # (B, max_knots)
        inc = F.softplus(raw) + 1e-6
        cum = torch.cumsum(inc, dim=-1)
        knots = torch.zeros_like(cum)
        for b in range(B):
            K = int(num_knots[b])
            knots[b, :K] = cum[b, :K] / (cum[b, K - 1] + 1e-8)   # last -> 1.0 (known head limitation)
        return knots

    @torch.no_grad()
    def predict(self, points, num_knots):
        """points:(N,dim) single sample -> (K,) interior knots in (0,1)."""
        self.eval()
        pts = points.unsqueeze(0)
        mask = torch.zeros(1, pts.shape[1], dtype=torch.bool, device=pts.device)
        nk = torch.tensor([num_knots], device=pts.device)
        return self.forward(pts, mask, nk)[0, :num_knots]


def collate_3d(batch):
    """batch: list of (points(N,dim), num_knots). Pads to max N."""
    dim = batch[0][0].shape[1]
    maxN = max(p.shape[0] for p, _ in batch)
    B = len(batch)
    pts = torch.zeros(B, maxN, dim, dtype=torch.float64)
    mask = torch.ones(B, maxN, dtype=torch.bool)            # True = PAD
    nk = torch.zeros(B, dtype=torch.long)
    for i, (p, k) in enumerate(batch):
        n = p.shape[0]
        pts[i, :n] = p
        mask[i, :n] = False
        nk[i] = k
    return pts, mask, nk


# ----------------------------------------------------------------------
# Warm-start: transfer weights from a 2D checkpoint (skip the input layer)
# ----------------------------------------------------------------------
def warm_start_from(model, state_dict, verbose=True):
    """
    Copy weights from a (2D) checkpoint into this KnotTransformer3D where the key
    NAME matches and the SHAPE is compatible.

      * input_proj.*  -> always reinitialized (2D and 3D read different coords, and
                         the feature columns are not aligned: 3D inserts a z column).
      * pos_embed / knot_queries -> copied on the overlapping rows if sizes differ
                         (so different max_points / max_knots still transfer).
      * everything else (encoder, decoder, knot_head) -> copied only on exact shape.

    Returns (n_loaded, n_total, skipped[list of (name, reason)]).
    """
    tgt = model.state_dict()
    partial = {'pos_embed', 'knot_queries'}
    loaded, skipped = 0, []
    for k, v in tgt.items():
        if k.startswith('input_proj'):
            skipped.append((k, 'input layer (reinitialized by design)')); continue
        if k not in state_dict:
            skipped.append((k, 'absent in source')); continue
        s = state_dict[k]
        if tuple(s.shape) == tuple(v.shape):
            tgt[k] = s.clone(); loaded += 1
        elif k in partial and s.dim() == v.dim() and all(a <= b for a, b in zip(s.shape, v.shape)):
            sl = tuple(slice(0, a) for a in s.shape)
            vv = v.clone(); vv[sl] = s; tgt[k] = vv; loaded += 1
        elif k in partial and s.dim() == v.dim():
            sl = tuple(slice(0, min(a, b)) for a, b in zip(s.shape, v.shape))
            vv = v.clone(); vv[sl] = s[sl]; tgt[k] = vv; loaded += 1
        else:
            skipped.append((k, f'shape {tuple(s.shape)} != {tuple(v.shape)}'))
    # Safe fallback: if exact-name matching caught less than half the network, the
    # source likely uses different parent attribute names. Match by key-suffix + shape,
    # but ONLY when that (suffix, shape) is UNIQUE in the source (ambiguous matches such
    # as encoder-vs-decoder self_attn are skipped, never mis-mapped).
    if loaded < 0.5 * len(tgt):
        from collections import Counter
        def suf(k): return '.'.join(k.split('.')[1:])   # drop only the first token
        keys = [(suf(k), tuple(v.shape)) for k, v in state_dict.items()]
        counts = Counter(keys)
        lookup = {(suf(k), tuple(v.shape)): v for k, v in state_dict.items()}
        loaded, skipped = 0, []
        for k, v in tgt.items():
            if k.startswith('input_proj'):
                skipped.append((k, 'input layer (reinitialized by design)')); continue
            key = (suf(k), tuple(v.shape))
            if k in state_dict and tuple(state_dict[k].shape) == tuple(v.shape):
                tgt[k] = state_dict[k].clone(); loaded += 1
            elif counts.get(key, 0) == 1:
                tgt[k] = lookup[key].clone(); loaded += 1
            else:
                skipped.append((k, 'no unambiguous match'))
        if verbose:
            print(f"(suffix fallback, uniqueness-safe: {loaded}/{len(tgt)})")
    model.load_state_dict(tgt)
    if verbose:
        print(f"warm-start: transferred {loaded}/{len(tgt)} tensors from checkpoint")
        if loaded == 0:
            print("  WARNING: nothing transferred — the checkpoint's layer names do not")
            print("  match this model. Train from scratch, or share knot_transformer.py to align.")
        for k, why in skipped:
            print(f"    skip {k}  [{why}]")
    return loaded, len(tgt), skipped

'''
open('knot_transformer_3d.py','w').write(kt3d_src)
ic_src = r'''
"""Core compute for the interactive B-spline tool (widget-independent, testable)."""
import numpy as np, torch, matplotlib
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa
import bspline3d as b3
torch.set_default_dtype(torch.float64)

def curve_xyz(D, u, n=200):
    u = torch.as_tensor(u, dtype=torch.float64)
    su = torch.sort(u)[0].clamp(0.02, 0.98)
    P, U = b3.solve_control_points(D, su)
    t = torch.linspace(0, 1, n, dtype=torch.float64)
    C, _, _ = b3.eval_derivs(U, P, t)
    return C.detach().numpy()

QUAD = 100
def optimize_knots(D, K, steps=150, restarts=2, n_quad=QUAD):
    """Gradient-descend the bending energy. ALWAYS includes the chordal knots as a
    starting point, so the result can only match or beat chordal."""
    chord = b3.chordal_knots(D).numpy()
    inits = [chord] + [np.sort(np.random.uniform(0.15, 0.85, K)) for _ in range(restarts)]
    best_u = chord.copy()
    best_e = b3.safe_curvature(D, torch.tensor(chord).clamp(0.02, 0.98), n_quad)
    for init in inits:
        u = torch.tensor(np.sort(init).clip(0.03, 0.97), requires_grad=True)
        opt = torch.optim.Adam([u], lr=2e-2)
        for _ in range(steps):
            su = torch.sort(u)[0].clamp(0.02, 0.98)
            try: e = b3.bending_energy(D, su, n_quad=n_quad)
            except Exception: break
            if not torch.isfinite(e): break
            opt.zero_grad(); e.backward()
            torch.nn.utils.clip_grad_norm_([u], 1.0); opt.step()
        uu = torch.sort(u.detach())[0].clamp(0.02, 0.98)
        ev = b3.safe_curvature(D, uu, n_quad)
        if np.isfinite(ev) and ev < best_e:
            best_e, best_u = ev, uu.numpy()
    # hard guarantee: never return something worse than chordal
    ec = b3.safe_curvature(D, torch.tensor(chord).clamp(0.02, 0.98), n_quad)
    if not (best_e < ec):
        best_u = chord
    return best_u

def compute_figure(points, dim, K, source="Energy-optimized (target)", model=None, DEVICE="cpu"):
    """points: (N,dim) array. Returns (fig, kappa_chordal, kappa_cmp)."""
    D = torch.tensor(np.asarray(points, dtype=float))
    assert D.shape[0] == K + 2, f"need {K+2} points for {K} interior knots, got {D.shape[0]}"
    # left: chordal
    ck = b3.chordal_knots(D)
    kc = b3.safe_curvature(D, ck, QUAD)
    # right: comparison
    if source.startswith("Trained") and model is not None:
        pk = model.predict(D.to(DEVICE), num_knots=K).cpu().numpy()
        pk = 0.02 + 0.96 * pk
        cmp_label = "Trained model"
    else:
        pk = optimize_knots(D, K)
        cmp_label = "Energy-optimized (target)"
    km = b3.safe_curvature(D, pk, QUAD)

    Cc, Cm = curve_xyz(D, ck), curve_xyz(D, pk)
    Dn = D.numpy()
    fig = plt.figure(figsize=(12, 5.2))
    proj = "3d" if dim == 3 else None
    for pos, C, ttl, col, kap in [
        (121, Cc, "Chordal heuristic", "tab:blue", kc),
        (122, Cm, cmp_label, "tab:red", km)]:
        ax = fig.add_subplot(pos, projection=proj)
        if dim == 3:
            ax.plot(C[:,0], C[:,1], C[:,2], color=col, lw=2.5)
            ax.scatter(Dn[:,0], Dn[:,1], Dn[:,2], color="black", s=45, zorder=5)
            ax.set_zlabel("z")
        else:
            ax.plot(C[:,0], C[:,1], color=col, lw=2.5)
            ax.scatter(Dn[:,0], Dn[:,1], color="black", s=55, zorder=5)
            ax.set_aspect("equal", adjustable="datalim")
        ax.set_title(f"{ttl}\n\u03ba = {kap:.2f}", fontsize=11)
        ax.set_xlabel("x"); ax.set_ylabel("y")
    ratio = km / kc if kc else float("nan")
    fig.suptitle(f"{dim}D  |  {K} interior knots ({K+2} points)   "
                 f"chordal \u03ba={kc:.1f}   comparison \u03ba={km:.1f}   ratio={ratio:.3f}",
                 fontsize=12, y=1.02)
    fig.tight_layout()
    return fig, kc, km

'''
open('interactive_core.py','w').write(ic_src)
print('modules written')

modules written


## 2 · Imports


In [ ]:
import numpy as np
import ipywidgets as W
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import bspline3d as b3, interactive_core as ic
from knot_transformer_3d import KnotTransformer3D, warm_start_from
try:
    from google.colab import output as _c; _c.enable_custom_widget_manager()
except Exception: pass
print('ready')

ready


## 3 · (Optional) upload YOUR trained checkpoint

Skip this to use the energy-optimized target as the right-hand comparison. If you upload a
2D checkpoint it drives the 2D view; a 3D checkpoint drives the 3D view. Weights transfer
via the same warm-start loader (everything except the input layer).


In [ ]:
CKPT = {2: None, 3: None}      # state_dicts per dimension
MODELS = {}                    # cached models per dimension
def get_model(dim):
    if dim in MODELS: return MODELS[dim]
    m = KnotTransformer3D(dim=dim, d_model=64, nhead=4, num_encoder_layers=3,
                          num_decoder_layers=3, dim_feedforward=256,
                          max_points=12, max_knots=10).to(DEVICE)
    if CKPT.get(dim) is not None:
        warm_start_from(m, CKPT[dim], verbose=False)
    MODELS[dim] = m; return m

def _upload(_):
    try:
        from google.colab import files
        up = files.upload()
        path = list(up.keys())[0]
        sd = torch.load(path, map_location=DEVICE)
        if isinstance(sd, dict) and 'model' in sd: sd = sd['model']
        # infer dim from the checkpoint's input_proj (in_feat = dim + 3)
        w = [v for k,v in sd.items() if k.endswith('input_proj.weight')]
        dim = int(w[0].shape[1] - 3) if w else 2
        CKPT[dim] = sd; MODELS.pop(dim, None)
        print(f'loaded checkpoint -> drives the {dim}D view')
    except Exception as e:
        print('upload skipped:', e)
btn_up = W.Button(description='Upload checkpoint', icon='upload')
btn_up.on_click(_upload); display(btn_up)

Button(description='Upload checkpoint', icon='upload', style=ButtonStyle())

## 4 · Interactive panel


In [ ]:
def random_points(dim, N, seed=None):
    r = np.random.default_rng(seed)
    return r.uniform(-0.5, 0.5, (N, dim))

def fmt_points(P):
    return '\n'.join(' '.join(f'{v:+.3f}' for v in row) for row in P)

def parse_points(txt, dim):
    rows = []
    for ln in txt.strip().splitlines():
        parts = ln.replace(',', ' ').split()
        if not parts: continue
        vals = [float(x) for x in parts[:dim]]
        if len(vals) < dim: vals += [0.0]*(dim-len(vals))
        rows.append(vals)
    return np.array(rows, dtype=float)

dim_w  = W.Dropdown(options=[('2D',2),('3D',3)], value=2, description='Dimension')
knot_w = W.IntSlider(value=4, min=1, max=10, description='Interior knots')
src_w  = W.Dropdown(options=['Energy-optimized (target)','Trained model'],
                    value='Energy-optimized (target)', description='Right panel')
pts_w  = W.Textarea(value=fmt_points(random_points(2, 6, 0)),
                    description='Points', layout=W.Layout(width='320px', height='170px'))
rand_w = W.Button(description='Randomize points', icon='refresh')
draw_w = W.Button(description='Update plot', button_style='primary', icon='play')
out    = W.Output()
status = W.HTML()

def refill(*_):
    N = knot_w.value + 2
    pts_w.value = fmt_points(random_points(dim_w.value, N, np.random.randint(1e6)))
    redraw()

def redraw(*_):
    dim, K = dim_w.value, knot_w.value
    with out:
        clear_output(wait=True)
        try:
            P = parse_points(pts_w.value, dim)
            if P.shape[0] != K + 2:
                status.value = (f"<b style='color:#b00'>Need {K+2} points for {K} "
                                f"interior knots — you have {P.shape[0]}.</b> "
                                f"Adjust the slider or the point list, or hit Randomize.")
                return
            model = get_model(dim) if src_w.value.startswith('Trained') else None
            note = ''
            if src_w.value.startswith('Trained') and CKPT.get(dim) is None:
                note = " <i>(no checkpoint for this dim — showing energy-optimized)</i>"
                model = None
            fig, kc, km = ic.compute_figure(P, dim, K,
                          source=('Trained model' if model is not None else 'Energy-optimized (target)'),
                          model=model, DEVICE=DEVICE)
            plt.show()
            status.value = (f"chordal κ = <b>{kc:.2f}</b> &nbsp; comparison κ = "
                            f"<b>{km:.2f}</b> &nbsp; ratio = <b>{km/kc:.3f}</b>{note}")
        except Exception as e:
            status.value = f"<b style='color:#b00'>error: {e}</b>"

def on_knots(*_):
    N = knot_w.value + 2
    P = parse_points(pts_w.value, dim_w.value)
    if P.shape[0] != N:
        pts_w.value = fmt_points(random_points(dim_w.value, N, np.random.randint(1e6)))
    redraw()

dim_w.observe(refill, names='value')
knot_w.observe(on_knots, names='value')
src_w.observe(redraw, names='value')
rand_w.on_click(refill)
draw_w.on_click(redraw)

controls = W.VBox([dim_w, knot_w, src_w, pts_w, W.HBox([rand_w, draw_w])])
display(W.HBox([controls, out]))
display(status)
redraw()

HTML(value='')

## 5 · Batch mode — one CSV, many maps

Paste (or upload) a CSV holding several point sets separated by `map N` lines, then hit
**Run all maps**. Each map is fit twice — chordal heuristic vs. the energy-optimized
target (or your trained model) — and you get a summary table, a grid of overlaid curves,
and `batch_results.csv`.

```
map 1
+0.123, -0.441
-0.302, +0.118
...            <- 6 to 8 rows
map 2
...
```

* Separator = any line starting with `map` (case-insensitive; `# map 3`, `MAP_3:`, `Map 3` all work).
* Values may be separated by commas, semicolons, tabs or spaces.
* A header row such as `x,y` is ignored.
* Interior knots are inferred per map: **K = points − 2**, so 6 points → 4 knots, 8 points → 6 knots.
  Different maps may have different point counts.
* For 3D, switch the Dimension dropdown and give three columns per row.

In [ ]:
# ==========================================================================
# 5 · Batch mode — one CSV, many maps   [refined: side-by-side view]
# --------------------------------------------------------------------------
# Changes from the original:
#   * plot_grid gains a `view` argument: 'side' (default) or 'overlay'.
#     'side'    -> two panels per map: chordal | comparison, shared axes.
#     'overlay' -> the original single-panel version, both curves together.
#   * A knot ruler strip under each map shows where the two schemes put
#     their knots in parameter space. This is usually the most informative
#     part: the curves often look similar while the knots differ a lot.
#   * Both panels of a pair share identical, square axis limits, so a
#     difference you see is a real difference and not autoscaling.
#   * New `view_w` dropdown wired into the existing controls.
#
# Everything above plot_grid (parse_maps, sample_csv, run_one) is unchanged.
# ==========================================================================
import re, math

SEP_RE = re.compile(r'^\s*[#>*\-]*\s*map', re.I)


def parse_maps(text, dim=2):
    """text -> [(label, (N,dim) float array), ...] in file order."""
    maps, rows, label = [], [], None

    def flush():
        if rows:
            maps.append((label or f'map {len(maps) + 1}', np.array(rows, float)))

    for raw in text.splitlines():
        line = raw.strip()
        if not line:
            continue
        if SEP_RE.match(line):
            flush()
            rows = []
            label = line.lstrip('#>*- ').strip().rstrip(':,').strip()
            continue
        cells = [c for c in re.split(r'[,;\s]+', line) if c]
        try:
            vals = [float(c) for c in cells[:dim]]
        except ValueError:
            continue                      # header row or stray text
        if len(vals) < dim:
            vals += [0.0] * (dim - len(vals))
        rows.append(vals)
    flush()
    return maps


def sample_csv(n_maps=12, dim=2, seed=0):
    """A ready-to-run example: n_maps blocks of 6-8 random points."""
    r = np.random.default_rng(seed)
    lines = []
    for i in range(1, n_maps + 1):
        lines.append(f'map {i}')
        for row in r.uniform(-0.5, 0.5, (int(r.integers(6, 9)), dim)):
            lines.append(', '.join(f'{v:+.3f}' for v in row))
        lines.append('')
    return '\n'.join(lines)


def run_one(P, source='Energy-optimized (target)', model=None):
    D = torch.tensor(np.asarray(P, dtype=float))
    K = D.shape[0] - 2
    ck = b3.chordal_knots(D)
    kc = float(b3.safe_curvature(D, ck, ic.QUAD))
    if source.startswith('Trained') and model is not None:
        pk = model.predict(D.to(DEVICE), num_knots=K).cpu().numpy()
        pk = 0.02 + 0.96 * pk
    else:
        pk = ic.optimize_knots(D, K)
    km = float(b3.safe_curvature(D, pk, ic.QUAD))
    return dict(n=int(D.shape[0]), K=int(K), kappa_chordal=kc, kappa_cmp=km,
                ratio=(km / kc if kc else float('nan')),
                knots_chordal=np.asarray(ck, dtype=float),
                knots_cmp=np.asarray(pk, dtype=float),
                curve_chordal=ic.curve_xyz(D, ck),
                curve_cmp=ic.curve_xyz(D, pk),
                pts=D.numpy())


# --------------------------------------------------------------------------
# plotting helpers
# --------------------------------------------------------------------------
CHORDAL_C = 'tab:blue'
COMPARE_C = 'tab:red'


def _square_limits(arrays, dim, pad=0.08):
    """A single cubic/square window covering every array. Keeps both panels
    of a pair on identical axes so visual differences are real."""
    stack = np.vstack([np.asarray(a, float).reshape(-1, dim) for a in arrays])
    lo, hi = stack.min(axis=0), stack.max(axis=0)
    centre = (lo + hi) / 2.0
    half = float(np.max(hi - lo)) / 2.0
    half = max(half, 1e-6) * (1.0 + pad)
    return centre - half, centre + half


def _draw_curve(ax, C, P, dim, colour, ls='-', lw=2.0):
    C = np.asarray(C, float)
    P = np.asarray(P, float)
    if dim == 3:
        ax.plot(C[:, 0], C[:, 1], C[:, 2], color=colour, lw=lw, ls=ls)
        ax.scatter(P[:, 0], P[:, 1], P[:, 2], color='black', s=16, zorder=5)
    else:
        ax.plot(C[:, 0], C[:, 1], color=colour, lw=lw, ls=ls)
        ax.scatter(P[:, 0], P[:, 1], color='black', s=20, zorder=5)


def _apply_limits(ax, lo, hi, dim):
    ax.set_xlim(lo[0], hi[0])
    ax.set_ylim(lo[1], hi[1])
    if dim == 3:
        ax.set_zlim(lo[2], hi[2])
        ax.set_zlabel('z', fontsize=7)
    else:
        ax.set_aspect('equal', adjustable='box')
    ax.tick_params(labelsize=6)


def _knot_ruler(ax, kc_vals, km_vals):
    """Thin strip: chordal knots above the line, comparison knots below."""
    kc_vals = np.asarray(kc_vals, float).ravel()
    km_vals = np.asarray(km_vals, float).ravel()
    both = np.concatenate([kc_vals, km_vals]) if kc_vals.size or km_vals.size else np.array([0.0, 1.0])
    lo, hi = float(both.min()), float(both.max())
    if hi - lo < 1e-9:
        lo, hi = lo - 0.5, hi + 0.5
    margin = 0.05 * (hi - lo)

    ax.axhline(0, color='0.75', lw=0.8, zorder=1)
    ax.vlines(kc_vals, 0.06, 0.85, color=CHORDAL_C, lw=1.6, zorder=2)
    ax.vlines(km_vals, -0.85, -0.06, color=COMPARE_C, lw=1.6, zorder=2)
    ax.set_xlim(lo - margin, hi + margin)
    ax.set_ylim(-1.1, 1.1)
    ax.set_yticks([])
    ax.tick_params(axis='x', labelsize=6, length=2, pad=1)
    for side in ('left', 'right', 'top'):
        ax.spines[side].set_visible(False)
    ax.spines['bottom'].set_visible(False)


def plot_grid(results, dim, ncols=3, panel=3.6, view='side', show_knots=True):
    """view='side'    -> chordal | comparison panels per map
       view='overlay' -> original single-panel overlay"""
    n = len(results)
    nrows = math.ceil(n / ncols)
    pair = 2 if view == 'side' else 1
    ruler_h = 0.55 if show_knots else 0.0

    fig = plt.figure(figsize=(panel * pair * ncols, (panel + ruler_h) * nrows))
    outer = fig.add_gridspec(nrows, ncols, hspace=0.45, wspace=0.28)

    for i, r in enumerate(results):
        row, col = divmod(i, ncols)
        heights = [panel, ruler_h] if show_knots else [panel]
        inner = outer[row, col].subgridspec(len(heights), pair,
                                            height_ratios=heights, hspace=0.12,
                                            wspace=0.15)

        Cc, Cm, P = r['curve_chordal'], r['curve_cmp'], r['pts']
        lo, hi = _square_limits([Cc, Cm, P], dim)
        proj = '3d' if dim == 3 else None

        if view == 'side':
            ax_l = fig.add_subplot(inner[0, 0], projection=proj)
            ax_r = fig.add_subplot(inner[0, 1], projection=proj)
            _draw_curve(ax_l, Cc, P, dim, CHORDAL_C, ls='--', lw=1.8)
            _draw_curve(ax_r, Cm, P, dim, COMPARE_C, ls='-', lw=2.0)
            _apply_limits(ax_l, lo, hi, dim)
            _apply_limits(ax_r, lo, hi, dim)
            ax_l.set_title(f"chordal   \u03ba {r['kappa_chordal']:.1f}", fontsize=8)
            ax_r.set_title(f"comparison   \u03ba {r['kappa_cmp']:.1f}"
                           f"   ({r['ratio']:.3f}\u00d7)", fontsize=8,
                           color=('tab:green' if r['ratio'] < 1 else 'tab:red'))
            head = ax_l
        else:
            ax = fig.add_subplot(inner[0, 0], projection=proj)
            _draw_curve(ax, Cc, P, dim, CHORDAL_C, ls='--', lw=1.8)
            _draw_curve(ax, Cm, P, dim, COMPARE_C, ls='-', lw=2.0)
            _apply_limits(ax, lo, hi, dim)
            ax.set_title(f"\u03ba {r['kappa_chordal']:.1f} \u2192 "
                         f"{r['kappa_cmp']:.1f}   ratio {r['ratio']:.3f}", fontsize=8)
            head = ax

        head.annotate(f"{r['label']}  ({r['n']} pts, K={r['K']})",
                      xy=(0, 1), xycoords='axes fraction',
                      xytext=(0, 22), textcoords='offset points',
                      fontsize=9, fontweight='bold', va='bottom')

        if show_knots:
            ax_k = fig.add_subplot(inner[1, :])
            _knot_ruler(ax_k, r['knots_chordal'], r['knots_cmp'])

    handles = [plt.Line2D([], [], color=CHORDAL_C, ls='--', lw=1.8, label='chordal'),
               plt.Line2D([], [], color=COMPARE_C, lw=2.0, label='comparison'),
               plt.Line2D([], [], color='black', ls='', marker='o', ms=4, label='data points')]
    fig.legend(handles=handles, loc='upper right', ncol=3, fontsize=9, frameon=False)
    fig.suptitle('knot ruler under each map: chordal above the line, comparison below',
                 fontsize=8, color='0.4', y=0.005, va='bottom')
    if dim == 2:
        fig.tight_layout(rect=[0, 0.015, 1, 0.965])
    else:
        # tight_layout can't measure 3D axes; rely on the gridspec spacing
        fig.subplots_adjust(top=0.94, bottom=0.04, left=0.04, right=0.97)
    return fig


# --------------------------------------------------------------------------
# batch driver
# --------------------------------------------------------------------------
def run_batch(text, dim=2, source='Energy-optimized (target)', model=None,
              ncols=3, show=True, save_csv='batch_results.csv', progress=None,
              view='side', show_knots=True):
    """Parse `text`, run every map, draw the grid, return (results, skipped)."""
    maps = parse_maps(text, dim)
    if not maps:
        raise ValueError('No maps found — need at least one line starting with "map".')

    results, skipped = [], []
    for j, (label, P) in enumerate(maps, 1):
        K = P.shape[0] - 2
        if not (1 <= K <= 10):
            skipped.append(f'{label}: {P.shape[0]} points (need 3-12)')
            continue
        r = run_one(P, source=source, model=model)
        r['label'] = label
        results.append(r)
        if progress is not None:
            progress.value = j
            progress.description = f'{j}/{len(maps)}'

    if not results:
        raise ValueError('Every map was skipped: ' + '; '.join(skipped))

    rows = [dict(map=r['label'], n_points=r['n'], K=r['K'],
                 kappa_chordal=round(r['kappa_chordal'], 4),
                 kappa_cmp=round(r['kappa_cmp'], 4),
                 ratio=round(r['ratio'], 4),
                 knots_chordal=' '.join(f'{v:.4f}' for v in r['knots_chordal']),
                 knots_cmp=' '.join(f'{v:.4f}' for v in r['knots_cmp']))
            for r in results]

    try:
        import pandas as pd
        df = pd.DataFrame(rows)
        if save_csv:
            df.to_csv(save_csv, index=False)
        if show:
            display(df.drop(columns=['knots_chordal', 'knots_cmp']))
    except ImportError:
        df = None
        if save_csv:
            import csv
            with open(save_csv, 'w', newline='') as f:
                w = csv.DictWriter(f, fieldnames=list(rows[0]))
                w.writeheader(); w.writerows(rows)
        if show:
            print(f"{'map':<10}{'n':>3}{'K':>3}{'chordal':>10}{'cmp':>10}{'ratio':>8}")
            for r in rows:
                print(f"{r['map']:<10}{r['n_points']:>3}{r['K']:>3}"
                      f"{r['kappa_chordal']:>10.2f}{r['kappa_cmp']:>10.2f}{r['ratio']:>8.3f}")

    ratios = np.array([r['ratio'] for r in results], float)
    if show:
        print(f"\n{len(results)} maps  |  mean ratio {ratios.mean():.4f}  "
              f"|  median {np.median(ratios):.4f}  |  best {ratios.min():.4f}  "
              f"|  improved on chordal: {(ratios < 1).sum()}/{len(ratios)}")
        if skipped:
            print('skipped -> ' + '; '.join(skipped))
        if save_csv:
            print(f'wrote {save_csv}')
        plot_grid(results, dim, ncols=ncols, view=view, show_knots=show_knots)
        plt.show()
    return results, skipped


# --------------------------------------------------------------------------
# widgets
# --------------------------------------------------------------------------
csv_w  = W.Textarea(value=sample_csv(12, 2, seed=0), description='CSV',
                    layout=W.Layout(width='360px', height='300px'))
bdim_w = W.Dropdown(options=[('2D', 2), ('3D', 3)], value=2, description='Dimension')
bsrc_w = W.Dropdown(options=['Energy-optimized (target)', 'Trained model'],
                    value='Energy-optimized (target)', description='Right panel')
view_w = W.Dropdown(options=[('Side by side', 'side'), ('Overlaid', 'overlay')],
                    value='side', description='View')
knot_v = W.Checkbox(value=True, description='Knot ruler', indent=False)
cols_w = W.IntSlider(value=3, min=1, max=4, description='Grid cols')
up_w   = W.FileUpload(accept='.csv,.txt', multiple=False, description='Load CSV')
samp_w = W.Button(description='Load sample', icon='refresh')
runb_w = W.Button(description='Run all maps', button_style='primary', icon='play')
prog_w = W.IntProgress(value=0, min=0, max=1, description='0/0',
                       layout=W.Layout(width='320px'))
bout   = W.Output()
bstat  = W.HTML()

_last = {'results': None, 'dim': 2}      # cache so redraw needn't recompute


def _read_upload(change=None):
    v = up_w.value
    items = list(v.values()) if isinstance(v, dict) else list(v)
    if not items:
        return
    raw = items[0]['content']
    csv_w.value = bytes(raw).decode('utf-8', 'replace')
    bstat.value = f"<b>loaded {items[0].get('name', 'file')}</b> — press <i>Run all maps</i>"


def _load_sample(_):
    csv_w.value = sample_csv(12, bdim_w.value, seed=int(np.random.randint(1e6)))
    bstat.value = '<b>sample regenerated</b>'


def _redraw_only(_=None):
    """Switching view or grid width only changes the picture, so reuse the
    cached results instead of re-running every map."""
    if _last['results'] is None:
        return
    with bout:
        clear_output(wait=True)
        plot_grid(_last['results'], _last['dim'], ncols=cols_w.value,
                  view=view_w.value, show_knots=knot_v.value)
        plt.show()


def _run(_):
    with bout:
        clear_output(wait=True)
        try:
            model = get_model(bdim_w.value) if bsrc_w.value.startswith('Trained') else None
            if model is not None and CKPT.get(bdim_w.value) is None:
                bstat.value = ('<i>no checkpoint for this dimension — '
                               'using the energy-optimized target</i>')
                model = None
            n_maps = len(parse_maps(csv_w.value, bdim_w.value))
            prog_w.max = max(n_maps, 1); prog_w.value = 0
            res, skipped = run_batch(csv_w.value, dim=bdim_w.value,
                                     source=bsrc_w.value, model=model,
                                     ncols=cols_w.value, progress=prog_w,
                                     view=view_w.value, show_knots=knot_v.value)
            _last['results'], _last['dim'] = res, bdim_w.value
            ratios = np.array([r['ratio'] for r in res])
            bstat.value = (f"ran <b>{len(res)}</b> maps &nbsp; mean ratio "
                           f"<b>{ratios.mean():.4f}</b> &nbsp; improved "
                           f"<b>{(ratios < 1).sum()}/{len(ratios)}</b>"
                           + (f" &nbsp; <span style='color:#b00'>skipped "
                              f"{len(skipped)}</span>" if skipped else ''))
        except Exception as e:
            bstat.value = f"<b style='color:#b00'>error: {e}</b>"


up_w.observe(_read_upload, names='value')
samp_w.on_click(_load_sample)
runb_w.on_click(_run)
view_w.observe(_redraw_only, names='value')
knot_v.observe(_redraw_only, names='value')
cols_w.observe(_redraw_only, names='value')

display(W.HBox([W.VBox([bdim_w, bsrc_w, view_w, knot_v, cols_w, csv_w,
                        W.HBox([up_w, samp_w]), runb_w, prog_w]), bout]))
display(bstat)

HTML(value='')